# The Fourier Basis of Digit Arithmetic — experiment compendium

Executable index to every experiment behind:

> **The Fourier Basis of Digit Arithmetic: Mechanistic Interpretability of
> Addition Circuits in Language Models**

For each experiment: **what it does**, **why we ran it**, **the exact command**,
and **what the result actually looked like** — including the hypothesis we
falsified.

> The companion notebook for the semantic-compass work lives in the separate
> `semantic-compass` repository. The two share no code.

---

## The claim in one paragraph

Language models represent decimal digits in a **Fourier basis of
$\mathbb{Z}/10\mathbb{Z}$**. At the computation layer the digit subspace
decomposes into exactly **nine** directions — two each for frequencies
$k = 1,2,3,4$ and one for $k = 5$ (parity). That subspace is causally necessary:
zero it and addition collapses to chance, while a matched-dimension *random*
subspace ablation does nothing at all. Rotate its phase and the model's answer
shifts by a predictable amount mod 10.

### Execution tiers — read before running anything

**This notebook cannot download models.** Gemma-2B, Phi-3-mini and LLaMA-3.2-3B are gated on HuggingFace and need a GPU. In a sandboxed or air-gapped
environment that fetch fails, so the notebook is built in three tiers and only
Tier 0 is guaranteed to run anywhere:

| Tier | Needs | Runtime | What it gives you |
|---|---|---|---|
| **0 — Verify & Replay** | nothing but this repo | seconds | Math checks against synthetic oracles; **re-analysis of the recorded experimental artifacts committed here**. Real numbers from real runs. |
| **1 — Reproduce (small)** | HuggingFace access, CPU ok | minutes–hours | Recompute the GPT-2 results from scratch. |
| **2 — Reproduce (full)** | GPU + gated model access | days | The full cross-model grid. |

**Tier 0 is not a mock.** It parses the actual logs committed in this
repository and recomputes the statistics from them. If a Tier-0 cell prints a
number, that number came out of a real run.

Tier 1/2 cells are **inert by default**: they print the command they would run
and stop. Set `RUN_TIER` below to arm them, after the preflight cell confirms
the environment can support it.


### Two structural cautions

**Everything here is Tier 2.** Recorded runtime is 30–60 min for Step 1 alone
and 1–2 h for the Step 3 sweep, *per model*. Budget days, not hours. The Tier 0
cells verify the mathematics and replay a committed Pythia-1.4B run.

**`ARITHMETIC_CIRCUIT_PLAN.md` also documents a superseded route** — "What
Already Exists", "Phase 0", and `SUPPLEMENTARY SCRIPTS` S5 ("Old Pipeline") —
built on a mask-learning dependency that has since been removed. **Commands in
those sections will not run.** They are retained as a record of how the work
developed. Phases A–F below are the paper's actual method.

## 0 · Configuration

In [ ]:
import json, os, re, sys, textwrap, subprocess
from pathlib import Path

# ─────────────────────────────────────────────────────────────── settings
RUN_TIER = 0        # 0 = verify + replay | 1 = also recompute small model | 2 = full grid
DEVICE   = "auto"   # "auto" | "cpu" | "cuda" | "mps"

# ────────────────────────────────────────────────── repository resolution
# Looks for markers unique to this repo, so the notebook works whether it sits
# inside the combined tree or in the standalone arithmetic-circuit-discovery repository.
def find_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "ARITHMETIC_CIRCUIT_PLAN.md").is_file() and (d / "experiments").is_dir():
            return d
    raise SystemExit(
        "Could not locate the repository root.\n"
        "Expected an ancestor directory containing 'ARITHMETIC_CIRCUIT_PLAN.md' and 'experiments/'."
    )

REPO = find_root(Path.cwd().resolve())
FR = REPO / "fourier_results"     # recorded artifacts; Tier 0 reads these

print(f"repository   : {REPO}")
print(f"python files : {sum(1 for _ in REPO.rglob('*.py'))}")
print(f"artifacts    : FR -> {len(list(FR.glob('*'))) if FR.is_dir() else 0} files")
print(f"tier         : {RUN_TIER}  "
      f"({'verify + replay recorded artifacts' if RUN_TIER == 0 else 'live recomputation ARMED'})")

## 0.1 · Preflight

The model-access probe is the one that matters: **if it fails, every Tier 2 cell
will fail**, which here is the entire live pipeline.

In [ ]:
import importlib.util

def have(mod: str) -> bool:
    return importlib.util.find_spec(mod) is not None

print(f"python            : {sys.version.split()[0]}")
deps = ["torch", "numpy", "transformer_lens", "matplotlib", "pandas", "sklearn", "seaborn"]
missing = [d for d in deps if not have(d)]
for d in deps:
    print(f"  {d:<16}: {'yes' if have(d) else 'MISSING'}")

if have("torch"):
    import torch
    dev = ("cuda" if torch.cuda.is_available()
           else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
           else "cpu")
    if DEVICE != "auto":
        dev = DEVICE
    print(f"accelerator       : {dev}")
else:
    dev = "cpu"

# model access -- the gate for Tier 1 and 2
MODEL_ACCESS = False
try:
    import urllib.request
    urllib.request.urlopen("https://huggingface.co/gpt2/resolve/main/config.json", timeout=10)
    MODEL_ACCESS = True
except Exception as e:
    print(f"huggingface       : UNREACHABLE ({type(e).__name__})")
else:
    print("huggingface       : reachable")

print()
if missing:
    print(f"! missing packages: {', '.join(missing)}   ->  pip install {' '.join(missing)}")
if RUN_TIER > 0 and not MODEL_ACCESS:
    print("! RUN_TIER > 0 but HuggingFace is unreachable. Live cells WILL fail.")
    print("  Set RUN_TIER = 0 for verification and replay only.")
elif RUN_TIER == 0:
    print("Tier 0: replaying recorded artifacts. No model downloads required.")

def tier(n: int) -> bool:
    """Guard for live cells. True only if this environment can actually run them."""
    if RUN_TIER < n:
        print(f"[tier {n} cell -- inactive, RUN_TIER={RUN_TIER}. Command shown above, not executed.]")
        return False
    if not MODEL_ACCESS:
        print(f"[tier {n} cell -- SKIPPED, no model access.]")
        return False
    return True

def shell(cmd: str):
    print(textwrap.dedent(cmd).strip())

## 1 · Pipeline shape

**15 numbered steps in six phases**, strictly ordered — Step 1 determines the
`comp-layer` every later step needs.

| Phase | Steps | Question |
|---|---|---|
| **A — Discovery** | 1–3 | Where does arithmetic live, and is the code Fourier? |
| **B — Causal validation** | 4–7 | Is that subspace *necessary* and *sufficient*? |
| **C — Attribution** | 8–9 | Which heads and neurons write it? |
| **D — Mechanism** | 10–11 | *How* is addition performed? |
| **E — Generalisation** | 12–14 | Does it survive subtraction, multi-digit, new ranges? |
| **F — Multi-digit** | 15 | Gemma-specific extension |

### 1.1 — Step 0 · teacher-forced vs direct-answer

**Purpose.** Choose the prompt protocol *before* anything else: it changes the
unembedding basis and therefore every later measurement.

- **Teacher-forced** (default): `"Calculate 13 + 8 = 2"`, model predicts `1`.
  Valid when single-digit answer tokens 0–9 exist.
- **Direct-answer**: `"a + b = "`, model emits the whole answer as one token.
  **Required** when the tokenizer has single tokens for 0–198 — LLaMA-3.2-3B
  does. Pass `--direct-answer` to *every* later command.

**How to decide.** Run a handful of `"a + b = "` prompts; above ~90% accuracy in
that format, use direct-answer.

> **⚠ Getting this wrong is silent.** The pipeline runs and produces
> plausible-looking numbers. It surfaces much later as near-zero unembed-patching
> transfer, which reads like a scientific null rather than a configuration error.
> `diagnose_unembed_direct.py` exists because we hit exactly this.

In [ ]:
print("Register a new model in experiments/arithmetic_circuit_scan_updated.py:\n")
print('    MODEL_MAP = {')
print('        "phi-3":      "microsoft/Phi-3-mini-4k-instruct",')
print('        "gemma-2b":   "google/gemma-2-2b",')
print('        "llama-3b":   "meta-llama/Llama-3.2-3B",')
print('        "your-model": "org/model-name",   # <- add here')
print('    }\n')
print("and its layer defaults in experiments/eigenvector_dft.py:\n")
print('    readout_defaults = {"gemma-2b": 25, "phi-3": 31, "llama-3b": 27}')
print('    comp_defaults    = {"gemma-2b": 19, "phi-3": 26, "llama-3b": 20}\n')
print("comp-layer is DISCOVERED by Step 1; those defaults only cache what we found.")

reg = REPO / "src" / "utils" / "model_registry.py"
print(f"\nshipped registry: {reg.relative_to(REPO)}  ({'present' if reg.exists() else 'MISSING'})")

## 2 · Step 1 — layer scan + unembed patching **(must run first)**

| | |
|---|---|
| **Script** | `experiments/arithmetic_circuit_scan_updated.py` |
| **Purpose** | Find the **comp-layer** and **readout-layer**. Every later step depends on this. |
| **Key functions** | `run_layer_scan`, `compute_unembed_basis[_direct_answer]`, `run_patching_experiment`, `compute_fisher_matrix`, `compute_contrastive_fisher`, `filter_correct_*` |
| **Output** | `mathematical_toolkit_results/arithmetic_scan_<model>.json` |
| **Runtime** | 30–60 min per model |

**Method.** Sweep every layer; patch activations between digit-pairs; measure
transfer. Then patch only a *subspace* (unembed-aligned / Fisher / random) to
measure effective dimensionality.

**What to look for.** First layer above 80% transfer → **comp-layer**; last layer
at ~100% → **readout-layer**; 9D unembed patching should capture most transfer at
readout.

**Expected outcome.** comp-layer ≈ **L19** (Gemma-2B), **L26** (Phi-3),
**L20** (LLaMA-3.2-3B); readout ≈ L25 / L31 / L27.

> **⚠ A dissociation you will hit mid-stack.** At Gemma L21, Fisher patching
> transfers **85%** but unembed patching only **30%**. The digit code is
> *gradient-visible but not yet output-aligned*; Fisher alignment and unembedding
> rotation are separate processes converging only at output layers. **A low
> unembed number mid-stack is a real finding, not a bug.**

In [ ]:
cmd = """
python experiments/arithmetic_circuit_scan_updated.py \\
    --model gemma-2b --device cuda --n-per-digit 100 --n-test 150
# LLaMA-3.2-3B additionally needs:  --direct-answer
"""
shell(cmd)
print("\nExpected comp-layer / readout-layer, from our runs:")
for m, comp, ro in [("gemma-2b", 19, 25), ("phi-3", 26, 31), ("llama-3b", 20, 27)]:
    print(f"    {m:<12} comp-layer L{comp:<3} readout-layer L{ro}")
if tier(2):
    subprocess.run(cmd, shell=True, cwd=REPO, check=True)

## 3 · Step 2 — eigenvector DFT · **is it really a Fourier basis?**

| | |
|---|---|
| **Script** | `experiments/eigenvector_dft.py` → plots by `plot_eigenvector_dft.py` |
| **Purpose** | Test whether the digit encoding is a *perfect* Fourier basis of $\mathbb{Z}/10\mathbb{Z}$. |
| **Method** | Take each SVD direction's 10-element digit-score vector, DFT it, ask which frequency dominates. |
| **Output** | `mathematical_toolkit_results/eigenvector_dft_<model>.json` |
| **Runtime** | 15–30 min |

**The prediction.** A perfect basis assigns exactly **2 directions to each of
k=1,2,3,4** and **1 to k=5** — $\cos$ and $\sin$ pair for every frequency except
the Nyquist frequency $k=5$, which is real-valued. Total: **9**.

**Expected outcome.** `"PERFECT FOURIER BASIS"` with mean purity > 50%.

> **⚠ This is the gate.** If the assignment is not 2/2/2/2/1, the model may not
> use a Fourier encoding at all and the rest of the pipeline is not meaningful.
> Before concluding that, check digit balance in your sample and raise `n`.

The cell below derives the number 9 from first principles and verifies the basis
is orthogonal and complete — no model required.

In [ ]:
import numpy as np

N = 10
print("Fourier basis of Z/10Z -- degrees of freedom per frequency\n")
print(f"{'k':>3}  {'basis functions':<24}{'dims':>6}   note")
print("-" * 66)
total = 0
for k in range(0, N // 2 + 1):
    if k == 0:
        dims, fns, note = 0, "constant", "DC -- removed by centring"
    elif k == N // 2:
        dims, fns, note = 1, "cos(2*pi*5*d/10)", "Nyquist: sin() vanishes"
    else:
        dims, fns, note = 2, f"cos, sin(2*pi*{k}*d/10)", "conjugate pair"
    total += dims
    print(f"{k:>3}  {fns:<24}{dims:>6}   {note}")
print("-" * 66)
print(f"{'':>3}  {'TOTAL':<24}{total:>6}   <- the '9D Fourier subspace'")
assert total == 9

d = np.arange(N)
B = np.array([np.cos(2*np.pi*k*d/N) for k in range(1, 5)]
             + [np.sin(2*np.pi*k*d/N) for k in range(1, 5)]
             + [np.cos(np.pi*d)])
G = B @ B.T
print(f"\nGram off-diagonal max        : {np.abs(G - np.diag(np.diag(G))).max():.2e}  (orthogonal)")
print(f"rank of the 9 basis functions: {np.linalg.matrix_rank(B)}  (expect 9)")
print(f"rank of [basis ; constant]   : {np.linalg.matrix_rank(np.vstack([B, np.ones(N)]))}  (expect 10 = complete)")

In [ ]:
cmd = "python experiments/eigenvector_dft.py --model gemma-2b --comp-layer 19 --device cuda"
shell(cmd)
print('\nLook for: "PERFECT FOURIER BASIS", mean purity > 50%,')
print("          frequency assignment 2/2/2/2/1 for k=1,2,3,4,5.")
if tier(2):
    subprocess.run(cmd, shell=True, cwd=REPO, check=True)

## 4 · Step 3 — Fourier layer sweep · where the structure is built

| | |
|---|---|
| **Script** | `experiments/fourier_decomposition.py` |
| **Purpose** | Track how Fourier energy accumulates layer by layer. |
| **Key functions** | `build_fourier_basis_functions`, `fourier_decomposition`, `per_neuron_fourier_analysis`, `run_fourier_at_layer` |
| **Runtime** | 1–2 h for a full sweep |

**Expected outcome.** Energy builds from early layers and **explodes by 2–3
orders of magnitude** at the computation layers — active amplification, not
passive propagation. This produces `energy_explosion.png`.

The related bottom-up head scan (`src/analysis/fourier_discovery.py`) produced
the committed Pythia-1.4B results replayed below — **real recorded output**.

In [ ]:
runs = sorted(FR.glob("fourier_results_*.json"))
print(f"recorded runs: {[p.name for p in runs]}\n")

fr = json.loads(runs[-1].read_text())
cfg = fr["config"]
print(f"model             : {fr['model_key']}")
print(f"layers analysed   : {fr['n_layers_analyzed']}")
print(f"significant heads : {fr['n_significant_heads']}  "
      f"(power-ratio threshold {cfg['fourier']['head_power_ratio_threshold']})")
print(f"prompt template   : {cfg['arithmetic']['prompt_template']!r}"
      f"  operands {cfg['arithmetic']['operand_range_start']}-{cfg['arithmetic']['operand_range_end']}")

from collections import Counter
heads = fr["head_results"]
freqs = Counter(h["dominant_frequency"] for h in heads.values())
print(f"\ndominant frequency across {len(heads)} significant heads:")
for k, n in sorted(freqs.items()):
    print(f"    k={k}: {n:>4} heads  {'#' * int(60 * n / len(heads))}")

ratios = np.array([h["power_ratio"] for h in heads.values()])
print(f"\npower ratio: median {np.median(ratios):.2f}, max {ratios.max():.2f}")
print("\nstrongest heads:")
for name, h in sorted(heads.items(), key=lambda kv: -kv[1]["power_ratio"])[:5]:
    print(f"    {name:<8} L{h['layer']:<3} H{h['head']:<3} k={h['dominant_frequency']}  ratio={h['power_ratio']:.2f}")

print("\nNote: k=1 (ordinal) dominates the head-level scan. Compare with section 5 --")
print("      the frequency that dominates by VARIANCE is not the causally important one.")

In [ ]:
cmd = 'python experiments/fourier_decomposition.py --model gemma-2b --layer-sweep "5,6,...,25" --device cuda'
shell(cmd)
if tier(2):
    subprocess.run(cmd, shell=True, cwd=REPO, check=True)

## 5 · Steps 4–7 — causal validation, and the ***k*=5 paradox**

| Step | Script | Tests |
|---|---|---|
| 4 | `fourier_knockout.py` | **Necessity** — zero the 9D subspace, measure damage |
| 5 | `fisher_phase_shift.py`, `fisher_patching.py` | **Sufficiency** — patch only the subspace |
| 6 | `fourier_phase_rotation.py` | **Steering** — rotate phase, predict shift mod 10 |
| 7 | `steering_improvements.py` | $W_U$-informed steering |

**Necessity — the cleanest result in the paper.** Zeroing the 9D subspace from
computation to readout drops accuracy to **near chance** in all three models. A
**matched-dimension random subspace ablation has *zero* effect** — perfect
specificity. Single-layer ablation does partial damage (11–77%), revealing a
distributed pipeline that actively maintains the information.

**Sufficiency.** At readout, standard Fisher at 10D transfers **85%** (Gemma,
Phi-3) and **100%** (LLaMA); contrastive Fisher at 9D transfers 83–100%. The two
subspaces agree to **>0.97 principal cosine**.

> ### ⚠ The *k*=5 paradox — a hypothesis we falsified
>
> At Gemma's computation layer $k=5$ (parity) is **the dominant SVD direction**:
> $\sigma = 135$, **71% of subspace variance**. The natural inference is that
> parity is central to the computation.
>
> **It is causally inert.** Ablating $k=5$ across L13–L25 costs **0.4–1.0%**
> accuracy. Ablating $k=1$ or $k=2$ costs **~40% each**.
>
> $k=5$ is an **epiphenomenal encoding** — strongly represented, not used for
> ones-digit computation. The causally necessary frequencies are $k=1$ (ordinal)
> and $k=2$ (mod-5).
>
> **The lesson, which generalises well beyond this paper: variance is not
> causation.** Had we ranked directions by singular value and stopped, we would
> have reported exactly the wrong mechanism.

**Steering, and its ceiling.** Coherent rotation at computation layers:
Gemma L19 **28%** exact, Phi-3 L26 **28%**, LLaMA L20 **69%**, with marked
backward-shift asymmetry. The bottleneck is the **encoding–readout gap**: the
Fourier–unembed overlap is only **8–11%**.

> **⚠ Readout layers are immune.** At Gemma L25 / LLaMA L27 every steering method
> lands at **≤10% (chance)**, and the overlap falls to **0.2%** at LLaMA L27. By
> readout the subspace has been absorbed into the unembedding-aligned
> representation. **If you steer at the readout layer you will measure nothing,
> and it is not a bug.**

In [ ]:
print("Gemma-2B computation layer -- variance rank vs causal importance\n")
print(f"{'frequency':<12}{'role':<16}{'sigma':>8}{'% variance':>12}{'ablation damage':>18}")
print("-" * 68)
for k, role, sig, var, dmg in [("k=5", "parity", "135", "71%", "0.4-1.0%"),
                               ("k=1", "ordinal", "-", "-", "~40%"),
                               ("k=2", "mod-5", "-", "-", "~40%")]:
    print(f"{k:<12}{role:<16}{sig:>8}{var:>12}{dmg:>18}")

print("\n  Ranked by variance : k=5 first.")
print("  Ranked by causation: k=5 LAST.")
print("\n  => Variance is not causation. Ablate before claiming a mechanism.")

In [ ]:
steps = [
    ("Step 4  necessity (knockout)",
     "python experiments/fourier_knockout.py --model gemma-2b --comp-layer 19 --device cuda"),
    ("Step 5  sufficiency (Fisher phase shift)",
     "python experiments/fisher_phase_shift.py --model gemma-2b --layers 19,25 --device cuda"),
    ("Step 6  steering (phase rotation)",
     "python experiments/fourier_phase_rotation.py --model gemma-2b --layers 19 --device cuda"),
    ("Step 7  W_U-informed steering",
     "python experiments/steering_improvements.py --model gemma-2b --layer 19 --device cuda"),
    ("        statistics over step 6",
     "python experiments/phase_rotation_statistics.py"),
]
for label, c in steps:
    print(f"# {label}\n{c}\n")
print("Reminder: run these at the COMPUTATION layer. At readout layers every")
print("          steering method returns chance-level results by construction.")
if tier(2):
    for label, c in steps:
        print(f"=== {label} ===")
        subprocess.run(c, shell=True, cwd=REPO, check=True)

## 6 · Steps 8–9 — component attribution

| | |
|---|---|
| **Scripts** | `fourier_head_attribution.py` (Step 8) · `neuron_trig_analysis.py` (Step 9) |
| **Purpose** | Identify which attention heads and MLP neurons *write* the Fourier subspace. |

**Expected outcome.** At Gemma L19, **809 / 9216 neurons (8.8%)** exceed 80%
frequency purity, dominated by $k=1$ and $k=5$ tunings. Produces
`neuron_frequency_tuning.png`.

> **⚠ Note the tension with §5.** $k=5$ is among the *most common* neuron tunings
> and is *causally inert*. **Prevalence of a tuning is not evidence of function
> either** — the same trap as variance ranking, one level down.

In [ ]:
for label, c in [
    ("Step 8  head/MLP attribution",
     "python experiments/fourier_head_attribution.py --model gemma-2b --comp-layer 19 --device cuda"),
    ("Step 9  per-neuron frequency tuning",
     "python experiments/neuron_trig_analysis.py --model gemma-2b --layer 19 --device cuda"),
]:
    print(f"# {label}\n{c}\n")

print("Expected (Gemma L19): 809/9216 neurons (8.8%) above 80% purity;")
print("                      dominant tunings k=1 and k=5.")
print("\nCaution: k=5 is prevalent AND causally inert (section 5). Prevalence != function.")

## 7 · Steps 10–11 — the computation mechanism

| | |
|---|---|
| **Scripts** | `cp_tensor_decomposition.py` (Step 10) · `carry_stratification.py` (Step 11) |
| **Purpose** | Show *how* addition is performed, not just where. |

**The mechanism.** Addition in a Fourier basis is **angle addition**. Step 10
decomposes the per-digit outer-product tensor
$T_{d,i,j} = \mathbb{E}[h_i h_j \mid \text{digit} = d]$ inside the Fourier
subspace and scores it against the product-to-sum identity
$$\cos\alpha\cos\beta = \tfrac{1}{2}\left[\cos(\alpha-\beta) + \cos(\alpha+\beta)\right]$$

**Expected outcome.** $\sigma^2$-weighted trigonometric identity score
**0.964** for Gemma — the strongest direct evidence that the model performs
angle addition rather than something merely correlated with it.

**Step 11** stratifies by carry vs no-carry: the ones-digit circuit should be
largely carry-invariant, since carry affects the tens digit.

In [ ]:
rng = np.random.default_rng(1)
a, b = rng.uniform(0, 2*np.pi, 10000), rng.uniform(0, 2*np.pi, 10000)
lhs = np.cos(a) * np.cos(b)
rhs = 0.5 * (np.cos(a - b) + np.cos(a + b))
print(f"max |cos(a)cos(b) - 0.5[cos(a-b)+cos(a+b)]| = {np.abs(lhs - rhs).max():.2e}   (identity holds)")

print("\nThe cos(a+b) term is literally the sum of the two operand angles.")
print("Finding it in the conditional-expectation tensor is what a Fourier adder looks like.\n")
print("Our measured sigma^2-weighted identity score (Gemma): 0.964  (1.0 = exact)\n")

for label, c in [
    ("Step 10  CP tensor decomposition",
     "python experiments/cp_tensor_decomposition.py --model gemma-2b --comp-layer 19 --device cuda"),
    ("Step 11  carry stratification",
     "python experiments/carry_stratification.py --model gemma-2b --comp-layer 19 --device cuda"),
    ("         CRT sanity check (S2)",
     "python experiments/crt_sanity_check.py --model gemma-2b --layer 19"),
]:
    print(f"# {label}\n{c}\n")

## 8 · Steps 12–15 — generalisation

| Step | Script | Question |
|---|---|---|
| 12 | `generalization_tests.py` | Subtraction, operand substitution, multi-digit |
| 13 | `fourier_umap.py` | UMAP view of the digit manifold |
| 14 | `multilayer_freq_ablation.py` | Per-frequency ablation across layer ranges |
| 15 | `multidigit_circuit.py` | Multi-digit circuit (Gemma) |

Step 14 produced the per-frequency damage numbers underpinning the *k*=5 paradox
in §5, so it is worth running despite being labelled "advanced".

> **⚠ UMAP is for illustration only.** A non-linear, stochastic embedding with no
> distance guarantees. A clean ring in UMAP is **not** evidence of circular
> structure — §3's DFT is. Never let a UMAP plot carry an argument.

In [ ]:
for label, c in [
    ("Step 12  generalisation",
     "python experiments/generalization_tests.py --model gemma-2b --comp-layer 19 --device cuda"),
    ("Step 13  UMAP (illustrative only)",
     "python experiments/fourier_umap.py --model gemma-2b --comp-layer 19 --device cuda"),
    ("Step 14  per-frequency ablation (drives the k=5 result)",
     "python experiments/multilayer_freq_ablation.py --model gemma-2b --comp-layer 19 --readout-layer 25 --device cuda"),
    ("Step 15  multi-digit circuit",
     "python experiments/multidigit_circuit.py --model gemma-2b --device cuda"),
]:
    print(f"# {label}\n{c}\n")

## 9 · Producing the paper figures

> **⚠ The six figures `paper/` needs are not in this repository.** `paper/main.tex`
> sets `\graphicspath{{../mathematical_toolkit_results/paper_plots/}}`, which is
> generated output that was never tracked. **`paper/` will not compile until you
> regenerate them.**

The plotting scripts read JSON written by the analysis steps, so each figure has
a prerequisite. The data steps emit **no images** — all rendering happens in the
three plotting scripts.

In [ ]:
figures = [
    ("layer_scan_curves.png",             "generate_paper_plots.py / generate_missing_plots.py", "Step 1  arithmetic_circuit_scan_updated.py"),
    ("fourier_heatmap_cross_model.png",   "generate_paper_plots.py",   "Step 3  fourier_decomposition.py"),
    ("energy_explosion.png",              "generate_paper_plots.py",   "Step 3  fourier_decomposition.py"),
    ("ablation_curves.png",               "generate_missing_plots.py", "Step 4  fourier_knockout.py + multilayer_freq_ablation.py"),
    ("neuron_frequency_tuning.png",       "generate_missing_plots.py", "Step 9  neuron_trig_analysis.py"),
    ("eigenvector_fourier_cross_model.png","plot_eigenvector_dft.py",  "Step 2  eigenvector_dft.py"),
]
print(f"{'figure':<38}{'plotted by':<52}{'needs'}")
print("-" * 132)
for fig_name, plotter, prereq in figures:
    print(f"{fig_name:<38}{plotter:<52}{prereq}")

plots_dir = REPO / "mathematical_toolkit_results" / "paper_plots"
have_figs = sorted(p.name for p in plots_dir.glob("*.png")) if plots_dir.is_dir() else []
print(f"\npaper_plots/ present: {have_figs or 'NONE -- paper/ will not compile yet'}")
print("(.gitignore was adjusted so this directory becomes trackable once populated.)")

## 10 · Reading these results responsibly

### 10.1 What is solid, what is refuted

| Claim | Evidence | Confidence |
|---|---|---|
| Digit codes are a 9D Fourier basis of ℤ/10ℤ | 2/2/2/2/1 assignment, purity >50% | **High** |
| That subspace is causally necessary | Knockout → chance; matched-dim random → zero effect | **High** — the specificity control is what makes it |
| Addition is angle addition | CP trig-identity score 0.964 (Gemma) | **High** |
| Fourier phase steering is practical | 28–69% exact; ≤10% at readout | **Provisional** — bounded by the 8–11% encoding–readout gap |
| Parity (*k*=5) drives the computation | 71% of variance, 0.4–1.0% ablation damage | **Refuted** — epiphenomenal |

### 10.2 Three ways to misuse this notebook

**Ranking directions by variance and stopping.** The *k*=5 paradox is the
cautionary tale: the largest singular direction, 71% of variance, is causally
inert. **Any direction that has not been ablated has not earned a causal claim.**

**Steering at the wrong layer.** At readout layers everything reads chance-level
because the subspace has been absorbed into the unembedding. Reporting that as a
negative result is a measurement error.

**Treating neuron-tuning prevalence as function.** §6: *k*=5 is both prevalent
and inert.

### 10.3 If a number here disagrees with your run

1. **Prompt protocol** — teacher-forced vs direct-answer (§1.1). The most common
   silent failure.
2. **Layer indices** — comp-layer is *discovered* by Step 1, not assumed. The
   cached defaults are for the exact checkpoints we used.
3. **Model revision** — "gemma-2-2b" today may not be the weights we ran. Pin a
   revision hash.
4. **Superseded sections of the plan** — "Phase 0" and S5 reference removed
   scripts and will not run.

## 10.4 Reproducibility ledger

In [ ]:
REPLAYED = sorted(FR.glob("*.json"))

import platform, datetime

print("=" * 68)
print("REPRODUCIBILITY LEDGER")
print("=" * 68)
print(f"timestamp     : {datetime.datetime.now().isoformat(timespec='seconds')}")
print(f"platform      : {platform.platform()}")
print(f"python        : {sys.version.split()[0]}")
try:
    import torch; print(f"torch         : {torch.__version__}  (cuda={torch.cuda.is_available()})")
except ImportError:
    print("torch         : not installed")
try:
    import transformer_lens; print(f"transformer_lens: {transformer_lens.__version__}")
except Exception:
    print("transformer_lens: not installed / version unavailable")
print(f"numpy         : {np.__version__}")
try:
    rev = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO,
                         capture_output=True, text=True).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain"], cwd=REPO,
                           capture_output=True, text=True).stdout.strip()
    print(f"git revision  : {rev}{' (dirty)' if dirty else ''}")
except Exception:
    print("git revision  : unavailable")
print(f"execution tier: {RUN_TIER}")
print(f"model access  : {'yes' if MODEL_ACCESS else 'NO -- tier 0 replay only'}")
print()
print("Artifacts replayed in this run:")
for p in REPLAYED:
    print(f"  {'ok ' if p.exists() else 'MISSING '}{p.relative_to(REPO)}")
print("=" * 68)